# Introduction to Queueing Theory: the Coffee Shop
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Session 8 · about 40 min.*

> **How to use this notebook.** Run the cells from top to bottom (Shift+Enter).
> Cells marked **✍️ TODO** are for you; a **✅ check** right after tells you if your answer is correct.
> Cells marked **🤖 AI prompt** contain a prompt you can paste into Gemini (the ✨ button in Colab),
> ChatGPT, Claude… **Always read and test the code the AI gives you.**

☕ **Story.** You manage a coffee shop. Customers arrive at random, wait in line and are served by your
baristas. How long do they wait? Is a faster (and more expensive) barista worth it? How many baristas do
you need at lunch? Queueing theory answers these questions.

**Plan (about 40 min), then Quiz 3.** Each part has a worked example **E** and a practice case **P**.
In Quiz 3 (Canvas, several attempts) you will use the functions of this notebook.

| Part | Topic | Time |
|------|-------|------|
| 1 | What is a queue? Kendall notation | 5 min |
| 2 | Arrivals and service: rates λ and μ | 6 min |
| 3 | Performance measures and Little's law | 6 min |
| 4 | One barista (M/M/1): formulas, simulation, *is the top barista worth it?* | 13 min |
| 5 | Several baristas (M/M/c): a staffing decision | 10 min |

In [ ]:
# @title Setup and helper functions (run this cell; no need to read it)
import base64, json, heapq
from math import factorial
import numpy as np
import matplotlib.pyplot as plt

_ANS = json.loads(base64.b64decode("eyJrZW5kYWxsXzEiOiAiTS9NLzEiLCAia2VuZGFsbF8yIjogIkQvTS8xIiwgImtlbmRhbGxfMyI6ICJNL00vNCIsICJQX2ludGVyYXJyaXZhbF9taW4iOiAzLjAsICJQX3NlcnZpY2VfbWluIjogMi41LCAibGl0dGxlX1dfbWluIjogMTIuMCwgIkUxX3JobyI6IDAuOCwgIkUxX0wiOiA0LjAsICJFMV9McSI6IDMuMiwgIkUxX1ciOiAwLjMzMzMzMzMzMzMzMzMzMzMsICJFMV9XcSI6IDAuMjY2NjY2NjY2NjY2NjY2NjYsICJQMV9yaG8iOiAwLjgzMzMzMzMzMzMzMzMzMzQsICJQMV9MIjogNS4wMDAwMDAwMDAwMDAwMDIsICJQMV9XcV9taW4iOiAxMi41LCAiUDFfV19taW4iOiAxNS4wLCAiUDFfV3FfbWluX2xhbTIyIjogMjcuNSwgIlBfdG9wX2Nvc3RfcmVndWxhciI6IDQ2LjAwMDAwMDAwMDAwMDAxLCAiUF90b3BfY29zdF90b3AiOiA0MC4wLCAiUDJfY19zdGFibGUiOiAzLCAiUDJfY190YXJnZXQiOiA0LCAiUDJfUHdhaXQiOiAwLjMxOTg1NjcwNDE5NjUyfQ=="))

def check(key, value, tol=0.01):
    """Compare your answer with the expected one (relative tolerance 1%)."""
    if value is None:
        print(f"⏳ {key}: not answered yet"); return
    exp = _ANS[key]
    if isinstance(exp, str):
        ok = str(value).replace(" ", "").upper() == exp.upper()
    else:
        ok = abs(float(value) - exp) <= tol * max(1.0, abs(exp))
    print(f"✅ {key}: correct" if ok else f"❌ {key}: not yet, check units (hours vs minutes) and formulas")

def show(res, unit="min"):
    """Print the performance measures of a queue; times converted from hours to minutes."""
    f = 60 if unit == "min" else 1
    rows = [("Utilization ρ", res.get("rho"), ""), ("P(wait)", res.get("Pwait"), ""),
            ("L  (customers in system)", res.get("L"), ""), ("Lq (customers in queue)", res.get("Lq"), ""),
            ("W  (time in system)", None if res.get("W") is None else res["W"] * f, unit),
            ("Wq (time in queue)", None if res.get("Wq") is None else res["Wq"] * f, unit)]
    for name, v, u in rows:
        if name.startswith("P(wait)") and v is None:
            continue
        print(f"  {name:<26} {'—' if v is None else f'{v:8.3f}'} {u}")

def draw_queue():
    fig, ax = plt.subplots(figsize=(9, 2.2))
    for i in range(5):
        ax.add_patch(plt.Circle((1.2 + i * 0.9, 1), 0.3, color="#9ecae1", ec="k"))
    ax.add_patch(plt.Rectangle((6.4, 0.45), 1.3, 1.1, color="#fdae6b", ec="k"))
    ax.text(7.05, 1, "server", ha="center", va="center")
    ax.annotate("", xy=(0.8, 1), xytext=(-0.3, 1), arrowprops=dict(arrowstyle="->", lw=2))
    ax.annotate("", xy=(9, 1), xytext=(7.8, 1), arrowprops=dict(arrowstyle="->", lw=2))
    ax.text(-0.3, 1.45, "arrivals (rate λ)"); ax.text(2.4, 0.2, "queue (waiting line)")
    ax.text(7.3, 1.75, "service (rate μ)", ha="center"); ax.text(8, 1.3, "departures")
    ax.set_xlim(-0.5, 9.5); ax.set_ylim(0, 2); ax.axis("off"); plt.show()

def simulate_mm1(lam, mu, n=50_000, seed=1):
    """Simulate n customers of an M/M/1 queue (FIFO). Returns the average W and Wq in hours."""
    rng = np.random.default_rng(seed)
    arr = np.cumsum(rng.exponential(1 / lam, n))
    srv = rng.exponential(1 / mu, n)
    start = np.empty(n); end = np.empty(n); free = 0.0
    for i in range(n):
        start[i] = max(arr[i], free); end[i] = start[i] + srv[i]; free = end[i]
    return dict(W=(end - arr).mean(), Wq=(start - arr).mean())

def simulate_mmc(lam, mu, c, n=50_000, seed=1):
    """Simulate n customers of an M/M/c queue (single FIFO line). Returns the average W and Wq in hours."""
    rng = np.random.default_rng(seed)
    arr = np.cumsum(rng.exponential(1 / lam, n)); srv = rng.exponential(1 / mu, n)
    servers = [0.0] * c; wq = np.empty(n); w = np.empty(n)
    for i in range(n):
        free = heapq.heappop(servers); start = max(arr[i], free)
        heapq.heappush(servers, start + srv[i]); wq[i] = start - arr[i]; w[i] = wq[i] + srv[i]
    return dict(W=w.mean(), Wq=wq.mean())

print("Setup done ✔")

---
## 1 · What is a queue?
A queueing system has **customers** that arrive, **wait** in a line if the servers are busy, are **served**,
and leave. Examples: a pharmacy counter, a call center, trucks at a loading dock, jobs on a printer.

In [ ]:
draw_queue()

**Kendall notation** `A/S/c` summarizes a queue:

| Symbol | Meaning | Common values |
|--------|---------|---------------|
| **A** | arrival process | **M** = Poisson arrivals (Markovian), **D** = deterministic (fixed interval), **G** = general |
| **S** | service times | **M** = exponential, **D** = constant, **G** = general |
| **c** | number of servers | 1, 2, … |

**E.** A bank: customers arrive at random (Poisson), service times are exponential, 3 tellers share one line
→ **M/M/3**.

### ✍️ P · Write the Kendall notation of each system (as text, e.g. `"M/M/2"`)

In [ ]:
# 1. A pharmacy with one pharmacist; random (Poisson) arrivals; exponential service times.
# 2. A clinic where patients arrive exactly every 10 minutes; exponential consultation times; one doctor.
# 3. An IT help desk with 4 agents; Poisson arrivals; exponential service times.
kendall_1 = None   # ✍️ TODO
kendall_2 = None   # ✍️ TODO
kendall_3 = None   # ✍️ TODO
check("kendall_1", kendall_1); check("kendall_2", kendall_2); check("kendall_3", kendall_3)

---
## 2 · Arrivals and service: rates λ and μ
* **λ (lambda)** = arrival **rate**, e.g. 12 customers per hour. With Poisson arrivals, the time between two
  arrivals is **exponential** with mean $1/\lambda$.
* **μ (mu)** = service **rate of one server**, e.g. 15 customers per hour. The mean service time is $1/\mu$.

⚠️ Rates are *per hour* here, so $1/\lambda$ is in **hours**. Multiply by 60 to get minutes.

**E.** The coffee shop in the afternoon: λ = 12 customers/hour → mean time between arrivals $= 1/12$ h $= 5$ min.
The simulation below draws 2,000 inter-arrival times and compares their histogram with the exponential curve.

In [ ]:
lam = 12
rng = np.random.default_rng(0)
gaps_min = rng.exponential(1 / lam, 2000) * 60
print(f"simulated mean gap: {gaps_min.mean():.2f} min   (theory: {60 / lam:.2f} min)")
x = np.linspace(0, 30, 200)
plt.hist(gaps_min, bins=40, density=True, alpha=0.6, label="simulated gaps")
plt.plot(x, lam / 60 * np.exp(-lam / 60 * x), "r", label="exponential density")
plt.xlabel("minutes between arrivals"); plt.legend(); plt.show()

### ✍️ P · Coffee shop morning rush: λ = 20 customers/hour, μ = 24 customers/hour. Mean times in **minutes**?

In [ ]:
P_interarrival_min = None   # ✍️ TODO: mean time between two arrivals (min)
P_service_min = None        # ✍️ TODO: mean service time (min)
check("P_interarrival_min", P_interarrival_min); check("P_service_min", P_service_min)

---
## 3 · Performance measures and Little's law

| Measure | Meaning |
|---------|---------|
| $\rho$ | utilization: fraction of time a server is busy |
| $L$, $L_q$ | average number of customers **in the system**, **in the queue** |
| $W$, $W_q$ | average time a customer spends **in the system**, **in the queue** |

**Little's law** links them, for *any* stable queue:
$$L = \lambda W, \qquad L_q = \lambda W_q, \qquad W = W_q + \frac{1}{\mu}.$$

**E.** On average 4 customers are in the coffee shop in the afternoon (L = 4) and λ = 12/h, so
$W = L/\lambda = 4/12$ h $= 20$ min.

### ✍️ P · Another coffee shop receives λ = 30 customers/hour and has on average L = 6 customers inside. W in minutes?

In [ ]:
little_W_min = None   # ✍️ TODO
check("little_W_min", little_W_min)

---
## 4 · One barista: the M/M/1 queue
One server (the barista), Poisson arrivals (λ), exponential service (μ). The queue is **stable** only if $\lambda < \mu$
(otherwise the line grows forever). Then:
$$\rho = \frac{\lambda}{\mu},\quad L = \frac{\rho}{1-\rho},\quad L_q = \frac{\rho^2}{1-\rho},\quad
W = \frac{1}{\mu-\lambda},\quad W_q = \frac{\rho}{\mu-\lambda}.$$

### ✍️ Complete the function `mm1` with these formulas (times in **hours**)

In [ ]:
def mm1(lam, mu):
    """Performance measures of an M/M/1 queue. Rates per hour; W and Wq in hours."""
    if lam >= mu:
        raise ValueError("Unstable queue: need lam < mu")
    rho = None   # ✍️ TODO
    L   = None   # ✍️ TODO
    Lq  = None   # ✍️ TODO
    W   = None   # ✍️ TODO
    Wq  = None   # ✍️ TODO
    return dict(rho=rho, L=L, Lq=Lq, W=W, Wq=Wq)

**E.** The coffee shop in the afternoon, λ = 12/h, μ = 15/h. If your function is right, all five checks pass.

In [ ]:
E1 = mm1(12, 15)
show(E1)
for k in ["rho", "L", "Lq", "W", "Wq"]:
    check("E1_" + k, E1[k])

**Theory vs simulation.** The formulas are *averages over a long time*. Simulating 50,000 customers gives
almost the same values:

In [ ]:
sim = simulate_mm1(12, 15)
print(f"W : simulated {sim['W'] * 60:6.2f} min   theory {E1['W'] * 60 if E1['W'] else float('nan'):6.2f} min")
print(f"Wq: simulated {sim['Wq'] * 60:6.2f} min   theory {E1['Wq'] * 60 if E1['Wq'] else float('nan'):6.2f} min")

**Why managers care: waiting explodes near full utilization.** The plot shows $W_q$ for μ = 15/h when λ grows.
Going from 80% to 90% utilization *doubles* the wait.

In [ ]:
mu = 15
lams = np.linspace(1, 14.6, 200)
wq = [lam / mu / (mu - lam) * 60 for lam in lams]
plt.plot(lams / mu, wq); plt.axvline(0.8, ls="--", c="gray"); plt.axvline(0.9, ls="--", c="gray")
plt.xlabel("utilization ρ = λ/μ"); plt.ylabel("Wq (minutes)"); plt.ylim(0, 120)
plt.title("M/M/1 waiting time vs utilization"); plt.show()

### ✍️ P · Coffee shop morning rush: λ = 20/h, μ = 24/h

In [ ]:
P1 = mm1(20, 24)
show(P1)
P1_rho = None         # ✍️ TODO: read from P1
P1_L = None           # ✍️ TODO
P1_Wq_min = None      # ✍️ TODO (minutes!)
P1_W_min = None       # ✍️ TODO (minutes!)
# What if 10% more customers arrive (λ = 22/h)?
P1_Wq_min_lam22 = None   # ✍️ TODO
for k, v in [("P1_rho", P1_rho), ("P1_L", P1_L), ("P1_Wq_min", P1_Wq_min),
             ("P1_W_min", P1_W_min), ("P1_Wq_min_lam22", P1_Wq_min_lam22)]:
    check(k, v)

### Is the top barista worth it?
You can pay more for a faster barista. Waiting customers also cost money (lost
goodwill, customers who leave). A simple way to compare options is the **total cost per hour**:
$$\text{cost} = \text{wage} + c_w \cdot L,$$
where $c_w$ is the cost of one customer spending one hour in the shop and $L$ comes from `mm1`.

**E.** λ = 20/h, $c_w$ = \$10 per customer-hour. Regular barista: μ = 24/h, \$15/h. Top barista: μ = 30/h, \$25/h.

In [ ]:
def total_cost(lam, mu, wage, wait_cost):
    """Wage + cost of the customers in the shop, per hour (needs your mm1 function)."""
    L = mm1(lam, mu)["L"]
    return None if L is None else wage + wait_cost * L

for name, mu, wage in [("regular", 24, 15), ("top", 30, 25)]:
    cost = total_cost(20, mu, wage, 10)
    print(f"{name:<8} ${cost:6.2f} per hour" if cost is not None else f"{name:<8} complete mm1 first")
# regular: 15 + 10 * 5 = 65 ; top: 25 + 10 * 2 = 45  -> the top barista is worth it

### ✍️ P · λ = 16/h, $c_w$ = \$8. Regular: μ = 20/h, \$14/h. Top: μ = 24/h, \$24/h. Which one is cheaper?

In [ ]:
P_top_cost_regular = None   # ✍️ TODO: total cost per hour with the regular barista
P_top_cost_top = None       # ✍️ TODO: total cost per hour with the top barista
check("P_top_cost_regular", P_top_cost_regular); check("P_top_cost_top", P_top_cost_top)

---
## 5 · Several baristas (M/M/c): a staffing decision
With $c$ identical servers sharing one line, the queue is stable if $\rho = \lambda/(c\mu) < 1$.
The probability that an arriving customer has to wait, $P(\text{wait})$, comes from the **Erlang-C** formula.
You do not need to derive it: the function below computes it, then Little's law gives the rest.

In [ ]:
def mmc(lam, mu, c):
    """Performance measures of an M/M/c queue. Rates per hour; W and Wq in hours."""
    a, rho = lam / mu, lam / (c * mu)
    if rho >= 1:
        return None                                   # unstable
    p0 = 1 / (sum(a**n / factorial(n) for n in range(c)) + a**c / factorial(c) / (1 - rho))
    p_wait = a**c / factorial(c) / (1 - rho) * p0     # Erlang-C
    Lq = p_wait * rho / (1 - rho)
    Wq = Lq / lam                                     # Little's law
    W = Wq + 1 / mu
    return dict(rho=rho, Pwait=p_wait, L=lam * W, Lq=Lq, W=W, Wq=Wq)

def staffing_table(lam, mu, c_max=7, target_wq_min=None):
    print(f"  c     ρ    P(wait)   Wq (min)")
    for c in range(1, c_max + 1):
        r = mmc(lam, mu, c)
        if r is None:
            print(f"  {c}   unstable"); continue
        ok = "  ← meets target" if target_wq_min and r["Wq"] * 60 <= target_wq_min else ""
        print(f"  {c}  {r['rho']:.3f}  {r['Pwait']:.3f}   {r['Wq'] * 60:8.2f}{ok}")

**E.** Downtown coffee shop at noon: λ = 40 customers/h, μ = 15 customers/h per barista,
target $W_q \le 3$ min. With 3 baristas the line is stable but customers wait almost 10 min;
**4 baristas** is the smallest team that meets the target. Check the simulation too.

In [ ]:
staffing_table(40, 15, target_wq_min=3)
sim = simulate_mmc(40, 15, 4)
print(f"\nc = 4: simulated Wq = {sim['Wq'] * 60:.2f} min   (theory {mmc(40, 15, 4)['Wq'] * 60:.2f} min)")

### ✍️ P · Campus coffee kiosk at lunch: λ = 30 customers/h, μ = 12 customers/h per barista, target Wq ≤ 2 min

In [ ]:
staffing_table(30, 12, target_wq_min=2)
P2_c_stable = None   # ✍️ TODO: smallest c for a stable queue
P2_c_target = None   # ✍️ TODO: smallest c with Wq <= 2 min
P2_Pwait = None      # ✍️ TODO: P(wait) with that number of agents
check("P2_c_stable", P2_c_stable); check("P2_c_target", P2_c_target); check("P2_Pwait", P2_Pwait)

---
## Quiz 3 (Canvas)
Keep this notebook open: `mm1`, `total_cost`, `mmc` and `staffing_table` answer every question.
Each attempt draws new questions; use the attempts to check your understanding.

### Take-aways
* A queue is described by its arrivals (λ), its service (μ) and its number of servers (c): Kendall `A/S/c`.
* Stability needs $\lambda < c\mu$. **Little's law** $L = \lambda W$ holds for any stable queue.
* Waiting time grows **non-linearly** with utilization: running close to 100% busy is expensive for customers.
* Adding servers is a trade-off between staff cost and waiting time; queueing formulas quantify it.

🤖 **AI prompt (optional, after the quiz):** *"Explain with a numerical example why one server twice as fast
(M/M/1 with rate 2μ) gives a shorter time in system than two servers of rate μ (M/M/2) for the same λ."*

**Further reading:** Hillier & Lieberman, *Introduction to Operations Research*, chapter on queueing theory.